# Week 8 · Wednesday: Vector index & collection management

Yesterday we stored the 30 invented help-centre articles once, in SQLite, in Chroma and in Pinecone, and searched them by cosine similarity. SQLite had to read every row to answer a question. Today we manage what holds the vectors: the index that makes a search fast, the collections and indexes that hold them, the updates that keep them current, the parts we split them into, and the distance measure that ranks them. Today:

1. **Index management:** why a search needs an index, how HNSW finds near vectors, and Chroma's index settings
2. **Collections:** a collection's lifecycle, and one model and one size per collection
3. **Upsert:** insert what is new and replace what exists, with ids taken from the source
4. **Partitioning vectors:** a collection per product, and when to partition
5. **Vector partitioning using metadata:** `where` filters and their operators
6. **Distance metrics:** cosine, dot product and Euclidean, by hand and in code
7. **The same in Pinecone:** an index's lifecycle, upsert, namespaces, filters and the metric

Sections 1 to 6 use Chroma and need no account. Section 7 runs against your own free Pinecone project and needs its API key saved in Colab's Secrets panel as `PINECONE_API_KEY`, with Notebook access switched on. Without a key, Runtime > Run all runs every Chroma part and stops at the first Pinecone cell, in 7.1.

Parts marked *Going deeper* are optional on a first pass: run their cells, then come back to them with the notes page.

**Studying on your own**

This notebook is written to be worked through alone as well as followed live. Allow about three hours for all of it. The ends of sections 2, 4 and 6 are good places to stop for a break.

| Sections | What you work through | About |
|---|---|---|
| setup, 1 and 2 | exact search timed, HNSW, Chroma's index settings, the collection lifecycle | 40 min |
| 3 and 4 | add against upsert, stable ids, a collection per product | 35 min |
| 5 and 6 | metadata filters, three distances by hand and in code | 45 min |
| 7 | the same ideas in Pinecone, ending with every index deleted | 40 min |
| Recap | the Quick check | 10 min |

How to work a subsection:

- Read the text above the code cell and answer its **Predict first** question before you run anything. Write the answer down: a guess you have committed to is what makes the result stick.
- Run the cell. Compare what it prints with your prediction and with the bullets under it. The numbers quoted in the text come from our own run. Chroma's results come from fixed articles and a fixed model, so yours should match or sit very close; the timings in 1.1 depend on the machine.
- Do the **Try changing** edit, see what moved, then put the value back and rerun the cell. Later cells rely on the original values.
- A **Common mistake** bullet names an error people often make at that step.
- **Work it by hand first** marks a paper exercise, in 6.1. Do it before you read on.
- A line such as "Check your answer" opens when you click it. Try first, then open it.
- Each section ends with **Before you move on**. If you cannot do one of its items, reread that subsection before going on.
- The Quick check at the end has its answers folded under it. Say each answer aloud before you open them: these are the questions an interviewer asks.
- If the runtime disconnects (Colab does this after a long idle spell), click the cell you were on and choose Runtime > Run before. The installs run again (about a minute), and 1.3 empties the Chroma folder before reopening it, so every later cell runs cleanly. Section 7 needs the names that 2.1, 3.1, 3.2 and 5.2 define, so it always needs Run before from the top.
- If Colab says "Too many sessions" when you open this notebook, close the runtime of the notebook you used before (Runtime > Disconnect and delete runtime in that tab) and try again.
- If you stop partway through section 7, run its 7.1 connect cell and then 7.6 before you leave, so no index stays in your Pinecone project.
- The [notes page for today](https://2494-gen-ai-cn.github.io/training/weeks/week-08/day-3-vector-index-management.html) covers the same ideas in prose, goes further in places (Pinecone's performance options, backups and rollback), and ends with a short check.

Run the next cell first. It installs the two pinned database clients, loads Tuesday's embedding model and the 30 articles, and defines the helpers we use all day. It ends with "Setup complete". On Colab the cell also prints red text that starts `ERROR: pip's dependency resolver` and names packages such as google-adk and tweepy; the installs still succeed, and this notebook does not use those packages. A warning about unauthenticated requests to the HF Hub, printed when the model loads, is harmless too.

In [ ]:
# ---------- Setup and plotting helpers (run once; no need to read live) ----------
import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:                                     # Colab ships neither client; Tuesday's 5.1 and 8.1 taught both
    %pip install -q chromadb==1.5.9
    %pip install -q pinecone==9.1.0

import os
import shutil
import time

import chromadb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pinecone
import sentence_transformers
from pinecone import Pinecone, ServerlessSpec
from sentence_transformers import SentenceTransformer

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Tuesday's embedding function: one row of 384 numbers of length 1 per text
encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")


def embed(texts):
    return encoder.encode(texts, normalize_embeddings=True).astype("float32")


# Tuesday's 30 help-centre articles, invented for this course (no real company or product)
KB = [
    {"id": "kb-001", "title": "Resetting a forgotten password", "text": "Choose Forgot password on the sign-in screen, type the email address you registered with, and open the link we send to pick a new password. The link expires after 30 minutes.", "category": "account", "product": "web", "updated": 2025},
    {"id": "kb-002", "title": "Turning on two-step verification", "text": "Two-step verification asks for a six-digit code from an authenticator app each time you sign in on a new device. Turn it on under Settings, Security, then scan the QR code shown.", "category": "account", "product": "web", "updated": 2024},
    {"id": "kb-003", "title": "Changing the email on your account", "text": "Open Settings, then Profile, and enter the new email address. We send a confirmation link to the new address, and the change takes effect only after you open that link.", "category": "account", "product": "web", "updated": 2023},
    {"id": "kb-004", "title": "Deleting your account", "text": "Deleting your account removes your profile, saved addresses and order history after a 14-day grace period. Request it under Settings, Privacy, Delete account; signing in again during the grace period cancels the request.", "category": "account", "product": "web", "updated": 2024},
    {"id": "kb-005", "title": "Signing out on every device", "text": "If you used a shared or lost phone, choose Sign out everywhere under Settings, Security. Every session ends at once, and each device asks for your password the next time it opens.", "category": "account", "product": "mobile", "updated": 2025},
    {"id": "kb-006", "title": "Signing in with fingerprint or face", "text": "On phones that support it, turn on biometric sign-in under Settings, Security in the mobile app. Your fingerprint or face data stays on the phone and is never sent to our servers.", "category": "account", "product": "mobile", "updated": 2026},
    {"id": "kb-007", "title": "Updating your display name", "text": "Your display name appears on reviews and messages. Change it under Settings, Profile, Display name; the new name shows everywhere within a few minutes and you may change it once every 30 days.", "category": "account", "product": "mobile", "updated": 2023},
    {"id": "kb-008", "title": "Ending a plan", "text": "To end a plan, open Billing, choose Manage plan and select Cancel renewal. Access continues until the current period ends, and no further amount is billed after that date.", "category": "billing", "product": "web", "updated": 2025},
    {"id": "kb-009", "title": "Two entries for one purchase", "text": "If your card statement lists the same purchase two times, the second line is usually a temporary authorisation hold. Your bank releases it within five business days; contact us if it remains after that.", "category": "billing", "product": "web", "updated": 2024},
    {"id": "kb-010", "title": "Downloading an invoice", "text": "Every order has a PDF invoice with tax shown separately. Open Orders, select the order, and choose Download invoice; invoices for business purchases also list your company name and tax number.", "category": "billing", "product": "web", "updated": 2026},
    {"id": "kb-011", "title": "Updating a payment card", "text": "Add or replace a card under Billing, Payment methods. The new card becomes the default for future orders and renewals, and an expired card is removed automatically after its last valid month.", "category": "billing", "product": "web", "updated": 2023},
    {"id": "kb-012", "title": "Refund timelines", "text": "Once a return is approved, the refund goes back to the original payment method. Cards usually show it within 5 to 10 business days; store credit appears in your wallet immediately.", "category": "billing", "product": "mobile", "updated": 2025},
    {"id": "kb-013", "title": "Applying a promo code", "text": "Enter a promo code in the Discount box at checkout before you pay. Only one code applies per order, codes cannot be added after an order is placed, and some exclude sale items.", "category": "billing", "product": "mobile", "updated": 2024},
    {"id": "kb-014", "title": "Paying in instalments", "text": "Orders over 100 dollars can be split into four equal instalments, taken every two weeks from the same card. Choose Pay in four at checkout; a missed instalment is retried after three days.", "category": "billing", "product": "mobile", "updated": 2026},
    {"id": "kb-015", "title": "Tracking an order that has not arrived", "text": "If a package has not arrived ten business days after dispatch, open Orders, select it and choose Report a problem. We trace it with the courier, then resend the items or refund you.", "category": "shipping", "product": "web", "updated": 2025},
    {"id": "kb-016", "title": "Delivery times and costs", "text": "Standard delivery takes three to seven business days and is free over 50 dollars. Express delivery arrives in one to two business days for a flat fee shown at checkout before you pay.", "category": "shipping", "product": "web", "updated": 2024},
    {"id": "kb-017", "title": "Changing a delivery address", "text": "You can change the delivery address while an order shows Processing. Open Orders, choose Edit address and save; once the status changes to Dispatched, contact the courier using the tracking number instead.", "category": "shipping", "product": "web", "updated": 2023},
    {"id": "kb-018", "title": "Shipping outside Canada", "text": "We ship to the United States and the United Kingdom. Import duties and taxes are collected by the courier on delivery and are not included in the price shown at checkout.", "category": "shipping", "product": "web", "updated": 2026},
    {"id": "kb-019", "title": "Returning an item", "text": "Most items can be returned within 30 days of delivery. Start a return under Orders, print the prepaid label, and drop the box at any post office; we email you when it is received.", "category": "shipping", "product": "mobile", "updated": 2024},
    {"id": "kb-020", "title": "Damaged items on arrival", "text": "If something arrives broken, take a photo of the item and the box within 48 hours and attach both to a problem report under Orders. We replace it at no cost without needing it back.", "category": "shipping", "product": "mobile", "updated": 2025},
    {"id": "kb-021", "title": "Picking up from a locker", "text": "Choose a pickup locker at checkout to collect your order yourself. You receive a code by text message when it is ready, and the locker holds it for five days before it is returned.", "category": "shipping", "product": "mobile", "updated": 2026},
    {"id": "kb-022", "title": "Fixing crashes on phones", "text": "If the mobile version quits unexpectedly, install the latest update from your phone's store, restart the phone, and clear cached data under Settings. Send a report from the Help menu if it continues.", "category": "technical", "product": "mobile", "updated": 2025},
    {"id": "kb-023", "title": "Turning on notifications", "text": "Allow notifications for our mobile app in your phone's system settings, then choose which alerts you want under Settings, Notifications: order updates, price drops and messages can each be switched separately.", "category": "technical", "product": "mobile", "updated": 2024},
    {"id": "kb-024", "title": "Supported browsers", "text": "The website supports the two latest versions of Chrome, Firefox, Safari and Edge. Older browsers may load pages slowly or show broken layouts, so update the browser before reporting a display problem.", "category": "technical", "product": "web", "updated": 2023},
    {"id": "kb-025", "title": "Pages loading slowly", "text": "Slow pages are usually caused by a weak network connection or a full browser cache. Try another network, clear the cache and cookies, and turn off extensions that block scripts before contacting support.", "category": "technical", "product": "web", "updated": 2024},
    {"id": "kb-026", "title": "Working offline in the mobile app", "text": "The mobile app keeps your saved lists and recent orders on the phone, so you can read them without a connection. Changes made offline sync automatically the next time the phone is online.", "category": "technical", "product": "mobile", "updated": 2026},
    {"id": "kb-027", "title": "Exporting your data", "text": "Request a copy of everything we store about you under Settings, Privacy, Export data. We prepare a ZIP file of JSON and CSV files and email a download link within 48 hours.", "category": "technical", "product": "web", "updated": 2025},
    {"id": "kb-028", "title": "Connecting a smart speaker", "text": "Link a smart speaker under Settings, Connected devices, then say the wake word followed by order status to hear where your latest order is. Unlink the speaker from the same page at any time.", "category": "technical", "product": "mobile", "updated": 2026},
    {"id": "kb-029", "title": "Error codes at checkout", "text": "Error E102 means the card issuer declined the payment and E205 means the session timed out. Refresh the page and try again, or choose another payment method if the same code repeats.", "category": "technical", "product": "web", "updated": 2023},
    {"id": "kb-030", "title": "Using the site with a screen reader", "text": "Every page works with common screen readers such as NVDA, JAWS and VoiceOver. Use the Skip to content link at the top of each page, and report any control that is not read aloud correctly.", "category": "technical", "product": "web", "updated": 2024},
]


def meta(a):
    """The metadata kept beside an article's vector: its display fields and the year it was updated."""
    return {"title": a["title"], "category": a["category"], "product": a["product"], "updated": a["updated"]}


def chroma_hits(result):
    """A Chroma query result (one query) as a table: id, distance, title, category, product."""
    rows = zip(result["ids"][0], result["distances"][0], result["metadatas"][0])
    return pd.DataFrame([{"id": i, "distance": round(d, 3), "title": m["title"], "category": m["category"],
                          "product": m["product"]} for i, d, m in rows])


def pinecone_hits(result):
    """A Pinecone query result as a table: id, score, title, category, product."""
    return pd.DataFrame([{"id": m.id, "score": round(m.score, 3), "title": m.metadata["title"],
                          "category": m.metadata["category"], "product": m.metadata["product"]}
                         for m in result.matches])


def wait_for_count(index, expected, namespace=None):
    """Poll the index stats every 2 seconds, for up to 60 seconds, until they show `expected` vectors."""
    for _ in range(30):
        stats = index.describe_index_stats()
        if namespace is None:
            count = stats.total_vector_count
        else:
            summary = stats.namespaces.get(namespace)
            count = summary.vector_count if summary else 0
        if count == expected:
            break
        time.sleep(2)
    print(f"{namespace or 'all namespaces'}: {count} vectors")


def plot_line(x, y, xlabel, ylabel, title=""):
    """Draw one line with markers, y against x, on one chart."""
    plt.figure(figsize=(8, 5))
    plt.plot(x, y, marker="o", markersize=6, linewidth=2)
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.grid(alpha=0.3)
    plt.show()


print(f"chromadb {chromadb.__version__}, pinecone {pinecone.__version__}, "
      f"sentence-transformers {sentence_transformers.__version__}, NumPy {np.__version__}, pandas {pd.__version__}")
print(f"{len(KB)} articles; embed gives shape {embed(['a test sentence']).shape}")
print("Setup complete")

## 1. Index management

Tuesday's SQLite search compared the question with every stored vector. That is fine for 30 articles and hopeless for 30 million. A **vector index** is a structure built beside the vectors so that a query reaches its near neighbours after visiting only a small share of them. This section times the search without one, then opens the index Chroma builds for every collection.

### 1.1 Why an index

**Exact search** (also called flat search) compares the query with every stored vector and keeps the closest. It always finds the true nearest vectors, and its cost grows with the number of rows.

The cell makes random vectors of 384 numbers, the size `embed` gives, at three sizes: 10,000, 100,000 and 400,000 rows. The meaning of the vectors does not matter for a timing, only how many there are.

- `rng.standard_normal((n, 384), dtype=np.float32)` draws the numbers as float32, 4 bytes each, half the memory of the 8-byte default. A row of 384 numbers is 1,536 bytes: about 15 MB, 154 MB and 614 MB for the three sizes.
- Dividing each row by its length gives it length 1, like the rows `embed` returns, so the dot product `vectors @ query` is the cosine similarity of every row with the query.
- `np.argsort(-scores)[:10]` lists the row positions from the highest score down and keeps the top 10. The minus sign puts the largest first.

**Predict first:** going from 10,000 rows to 100,000, roughly how much longer will one search take?

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
SIZES = [10_000, 100_000, 400_000]         # Try changing it: 800_000 (about 1.2 GB), or 200_000
query = rng.standard_normal(384, dtype=np.float32)
query /= np.linalg.norm(query)              # length 1, like every row embed returns

seconds = []
for n in SIZES:
    vectors = rng.standard_normal((n, 384), dtype=np.float32)    # float32: 4 bytes a number
    vectors /= np.linalg.norm(vectors, axis=1, keepdims=True)    # every row to length 1
    start = time.perf_counter()
    for _ in range(5):                                           # five searches, to steady the clock
        scores = vectors @ query                                 # one comparison per stored row
        top10 = np.argsort(-scores)[:10]                         # exact: every row was looked at
    seconds.append((time.perf_counter() - start) / 5)
    print(f"{n:>9,} rows  {vectors.nbytes / 1e6:5.0f} MB  {seconds[-1] * 1000:6.1f} ms per search")
del vectors, scores                                              # free the memory for the next cells

In [ ]:
plot_line(SIZES, [s * 1000 for s in seconds], "stored vectors", "milliseconds per search",
          "Exact search: the time grows with the rows")

- Roughly ten times longer: over our two Colab runs one search took about 3 to 4 ms at 10,000 rows, 36 to 39 ms at 100,000 and 123 to 140 ms at 400,000. Every row is compared once, so the time grows roughly in step with the rows, though the exact ratios moved between the two runs. Your times depend on the machine and change from run to run; the shape of the line does not.
- The chart rises in a roughly straight line: each extra row costs about the same. At 400 million rows the same search would take about a thousand times longer than at 400,000, for every single question.
- A vector index exists to stop paying for that full pass on every query.
- Try changing it: `400_000` to `800_000` in `SIZES`, and watch Colab's RAM gauge (top right) while it runs. The memory reaches about 1.2 GB and the time roughly doubles again (on a laptop, about 117 ms against 43 to 58 ms at 400,000; this size was not timed on Colab). If the runtime's memory cannot take it, use `200_000` instead. Put `400_000` back and rerun the cell before 1.2.
- **Common mistake:** trying a size far beyond the runtime's memory. Colab restarts the session and every name is lost. Click this cell, choose Runtime > Run before, and rerun it with `400_000`.

**In an interview:** without an index, a similarity search compares the query with every stored vector, so its time grows in proportion to the number of rows. A vector index lets a query reach its near neighbours after looking at only a small share of them.

### 1.2 Approximate nearest neighbours

**Approximate nearest-neighbour search (ANN)** finds vectors close to a query without comparing it with every stored vector, and accepts that a true neighbour is sometimes missed in return for speed.

**HNSW** (Hierarchical Navigable Small World, from a 2016 paper by Malkov and Yashunin) is the index Chroma builds for each collection. It is a stack of graphs. Each vector is a point linked to a few near neighbours. The bottom layer holds every vector; each layer above holds a smaller random subset, with longer links.

```
layer 2   E . . . . . . . . . . . . . . . P              few points, long links
          |                               |
layer 1   E . . . . . F . . . . . P . . . . . . R        more points, shorter links
                      |
layer 0   E . A . B . F . C . D . G . P . Q . R . S . T  every vector, linked to its near neighbours
```

A search walks it like this:

- Enter at the top layer, at the entry point `E`, where links are longest.
- Move to whichever linked neighbour is closest to the query, and keep moving until no neighbour is closer.
- Drop to the same point one layer down and walk again.
- On the bottom layer, return the closest points met.

The cost grows about logarithmically with the number of vectors, not in proportion to it. log₂ of 1,000 is about 10 and log₂ of 1,000,000 about 20.

The walk is greedy, so it can stop at a point whose neighbours are all farther from the query (a **local minimum**) while a closer vector sits elsewhere in the graph. That is why the index is approximate: fast, and almost always right.

Two settings trade speed for recall. More links per point (called **M**) and a longer list of candidates kept during the search (**efSearch**) raise recall and slow the search. Memory is set almost entirely by M.

**Predict first:** a collection grows from a thousand vectors to a million. Roughly how much longer does the HNSW walk become?

<details>
<summary>Check your answer</summary>

About twice as long. The walk's cost grows with log₂ of the number of vectors: about 10 steps' worth at a thousand and about 20 at a million. An exact search over the same growth would take a thousand times longer.

</details>

- **Common mistake:** hearing "approximate" as "wrong". The results an HNSW index returns are real, close vectors. It can miss a few of the true closest ones, and 1.4 measures how many.

**In an interview:** HNSW links each vector to a few near neighbours in several layers. A query enters at the top, walks greedily towards the query and drops a layer each time it stops improving, so the cost grows about logarithmically. The greedy walk can stop at a local minimum, which is why it is approximate.

### 1.3 Chroma's index settings

Chroma keeps its data in a folder on Colab's disk. The cell first empties that folder, so the notebook reruns from the top: a runtime restart keeps Colab's files, and a second run would otherwise meet every collection from the first one. `clear_system_cache()` makes Chroma forget the clients it opened earlier in the same session.

Chroma takes a collection's index settings when the collection is created, in `configuration={"hnsw": {...}}`. We create the collection `random_2k` (filled in 1.4) with one setting, `"space": "cosine"`, and print the whole configuration Chroma stored.

**Predict first:** we name only the space. Which other settings will print, and where do their values come from?

In [ ]:
chromadb.api.client.SharedSystemClient.clear_system_cache()   # forget clients from an earlier run
shutil.rmtree("chroma_wed", ignore_errors=True)  # start from an empty store, so the notebook reruns from the top
client = chromadb.PersistentClient(path="chroma_wed")
print("heartbeat:", client.heartbeat())
print("collections:", [c.name for c in client.list_collections()])

In [ ]:
random_2k = client.create_collection(
    "random_2k", configuration={"hnsw": {"space": "cosine"}}, embedding_function=None)
print(random_2k.configuration["hnsw"])

- The client cell prints a heartbeat (a timestamp: the database answered) and an empty list of collections: the folder `chroma_wed` starts empty.
- The configuration prints `{'space': 'cosine', 'ef_construction': 100, 'ef_search': 100, 'max_neighbors': 16, 'resize_factor': 1.2, 'sync_threshold': 1000}`. We set the space; every other value is Chroma's default.
- `space` is the distance measure. Without it a collection uses `l2`, which is why every collection this week names `cosine`.
- `max_neighbors` is Chroma's name for M, and `ef_search` its name for efSearch. `ef_construction` is the candidate list used while the graph is built. Chroma's documentation also lists `num_threads` (the CPU count) and `batch_size` (100).
- Some settings are fixed at creation and some can change later:

| Setting | Default | Can change later? |
|---|---|---|
| `space` | `l2` | no |
| `ef_construction` | 100 | no |
| `max_neighbors` | 16 | no |
| `ef_search` | 100 | yes, `collection.modify(configuration={"hnsw": {"ef_search": 200}})` |
| `num_threads`, `batch_size`, `sync_threshold`, `resize_factor` | Chroma's defaults | yes, the same way |

- Older code sets the space with `metadata={"hnsw:space": "cosine"}`. Chroma still accepts it for backwards compatibility but calls it deprecated.
- **Common mistake:** deleting the `shutil.rmtree` line as clutter. The first run works; a second Run all then fails with "Collection ... already exists".

**In an interview:** Chroma builds an HNSW index for each collection, configured at creation with `configuration={"hnsw": {...}}`. The space (the distance), `ef_construction` and `max_neighbors` are fixed once the collection exists; `ef_search` can be raised later to buy recall with slower queries.

### 1.4 How often the index misses

*Going deeper. The one idea to keep: an approximate index misses a few true neighbours, and a team measures that share, called recall, on its own data.*

2,000 random vectors of length 1 go into `random_2k`. Then 20 random queries are asked twice: once of numpy, which compares with all 2,000 and returns the true top 10, and once of the index. **Recall@10** is the share of the true top 10 the index returned, over all 20 queries: 200 true neighbours in all.

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
stored = rng.standard_normal((2_000, 384), dtype=np.float32)
stored /= np.linalg.norm(stored, axis=1, keepdims=True)
queries = rng.standard_normal((20, 384), dtype=np.float32)
queries /= np.linalg.norm(queries, axis=1, keepdims=True)
random_2k.add(ids=[f"r{i}" for i in range(len(stored))], embeddings=stored)
print("stored:", random_2k.count())

found = 0
for q in queries:
    exact = set(np.argsort(-(stored @ q))[:10].tolist())                 # numpy's true top 10
    approx = random_2k.query(query_embeddings=[q], n_results=10)["ids"][0]   # the index's top 10
    found += len(exact & {int(i[1:]) for i in approx})
print(f"recall@10: {found} of {len(queries) * 10} true neighbours found = {found / (len(queries) * 10):.2f}")

- On our runs the index found about 180 of the 200 true neighbours: a recall of about 0.9 (0.88 to 0.92 across our laptop and Colab runs). Earlier runs of the same cell on a laptop printed between 0.92 and 0.96. HNSW's graph is built slightly differently each time, so the share moves a little from run to run.
- Random vectors of 384 numbers are a hard case: they all sit at about the same distance from each other, so many points are nearly tied. Real text embeddings have more structure.
- Every result the index returned is a genuinely close vector; the misses are true neighbours it never reached.
- To raise recall a team raises `ef_search` with `modify`, measures again, and stops when recall is high enough for the job.

**Before you move on**, check that you can:

- say how an exact search's time changes when the number of rows grows tenfold, and why;
- describe the HNSW walk in four steps, and say why it can miss a true neighbour;
- name the HNSW settings Chroma prints and say which of them are fixed at creation;
- work out a recall from "found 180 of 200 true neighbours".

## 2. Collections

A **collection** is a named set of records in Chroma, each an id, a vector and optional text and metadata, with its own HNSW index. A Chroma folder can hold many collections. This section builds the day's main collection, runs every lifecycle call on a scratch one, and shows the one rule every collection keeps.

### 2.1 The collection lifecycle

First `kb_articles`, Tuesday's collection in one cell: the 30 articles embedded with `embed`, a cosine collection with `embedding_function=None` (Chroma would otherwise attach its own copy of a model and embed any text sent without a vector), and one `add` with the ids, vectors, texts and metadata. `meta(a)`, defined in the setup cell, keeps an article's title, category, product and year beside its vector.

**Predict first:** what will the cell print for the shape of `kb_vectors` and for `count()`?

In [ ]:
kb_vectors = embed([a["text"] for a in KB])
kb = client.create_collection(
    "kb_articles", configuration={"hnsw": {"space": "cosine"}}, embedding_function=None)
kb.add(ids=[a["id"] for a in KB], embeddings=kb_vectors,
       documents=[a["text"] for a in KB], metadatas=[meta(a) for a in KB])
print(kb_vectors.shape, kb.count())

- `(30, 384) 30`: 30 rows of 384 numbers went in, and the collection holds 30 records.
- `kb` is the Python name of the collection `kb_articles`. Sections 3 and 5 update and query it.

Now the lifecycle, on a scratch collection, `kb_scratch`, so `kb_articles` stays untouched. It gets the first four articles. Then we try to create it a second time, ask for it with `get_or_create_collection`, and list the collections.

- `create_collection` makes a new collection and refuses a name that exists.
- `get_or_create_collection` returns the collection when the name exists and creates it when it does not. Since Chroma 0.5.11 it ignores every other argument for a collection that exists, metadata included.
- `list_collections()` returns `Collection` objects (Chroma 1.0 and later), so we read their `.name`.
- Collection names are 3 to 512 characters, start and end with a letter or a digit, and may hold dots, dashes and underscores between.

**Predict first:** the second `create_collection("kb_scratch")` runs on a name that exists. Will it return the existing collection, replace it, or raise an error?

In [ ]:
scratch = client.create_collection(
    "kb_scratch", configuration={"hnsw": {"space": "cosine"}}, embedding_function=None)
scratch.add(ids=[a["id"] for a in KB[:4]], embeddings=kb_vectors[:4],
            documents=[a["text"] for a in KB[:4]], metadatas=[meta(a) for a in KB[:4]])
try:
    client.create_collection("kb_scratch", embedding_function=None)    # the name exists already
except Exception as err:
    print(f"{type(err).__name__}: {err}")
same = client.get_or_create_collection("kb_scratch", embedding_function=None)
print("get_or_create_collection returned the same collection:", same.id == scratch.id)
print("collections:", [c.name for c in client.list_collections()])

- The second create raises `InternalError: Collection [kb_scratch] already exists`. The `try`/`except` catches it so the notebook carries on, and prints the error's type and text.
- `get_or_create_collection` returns the same collection: the ids match, `True`.
- Three collections now: `kb_articles`, `kb_scratch` and `random_2k` from 1.3. The order of the list can differ from run to run; read the names, not the positions.

Next: open the scratch collection by name, count it, look at its first records, rename it and delete it.

- `get_collection(name, embedding_function=None)` opens an existing collection.
- `count()` returns the number of records. `peek()` returns the first 10; `peek(3)` the first three, with their ids, vectors, texts and metadata.
- `modify(name=...)` renames a collection. `delete_collection(name)` removes it with every record in it. Chroma's documentation says deleting is destructive and not reversible.

**Predict first:** after the delete, which collections will the last line list?

In [ ]:
opened = client.get_collection("kb_scratch", embedding_function=None)
print("count:", opened.count())
first = opened.peek(3)                       # the first 3 records: ids, vectors, texts, metadata
print("peek ids:", first["ids"], "| vector length:", len(first["embeddings"][0]))
opened.modify(name="kb_scratch_old")         # rename
print("after modify:", [c.name for c in client.list_collections()])
client.delete_collection("kb_scratch_old")   # destructive: there is no undo
print("after delete:", [c.name for c in client.list_collections()])

- `count: 4`, then the first three ids, `kb-001` to `kb-003`, each with a vector of 384 numbers.
- After `modify` the list shows `kb_scratch_old` in place of `kb_scratch`. After the delete it shows `kb_articles` and `random_2k`: the day's collection is still there.
- **Common mistake:** changing an argument in a `get_or_create_collection` line, such as the space, for a collection that exists, and expecting it to take effect. The existing collection comes back unchanged.

**In an interview:** in Chroma, `create_collection` makes a collection and raises if the name exists, and `get_or_create_collection` returns the existing one, ignoring its other arguments. `list_collections` lists them, `get_collection` opens one, `count` and `peek` inspect it, `modify` renames it and `delete_collection` removes it for good.

### 2.2 One model and one size per collection

The first vector added to a collection fixes its length: 384 for `kb_articles`. The cell tries to add one record whose vector has 768 numbers, the size a larger embedding model gives (Thursday meets one).

**Predict first:** after the add, what will `count()` print?

In [ ]:
wrong_size = np.zeros((1, 768), dtype="float32")        # the size a larger model gives
try:
    kb.add(ids=["kb-999"], embeddings=wrong_size, documents=["A vector from another model"],
           metadatas=[{"title": "Wrong size", "category": "technical", "product": "web", "updated": 2026}])
except Exception as err:
    print(f"{type(err).__name__}: {err}")
print("count:", kb.count())

- The add raises `InvalidArgumentError: Collection expecting embedding with dimension of 384, got 768`. Read it as written: the collection expects 384 numbers and was sent 768.
- `count: 30`: the add raised, so nothing was stored.
- A vector of the right length from another model is refused by nothing, and that is the dangerous case. Two models place texts in different spaces, so their numbers cannot be compared, and a search across them returns noise. A model change means a new collection, filled again from the source texts.
- **Common mistake:** thinking two models whose vectors both have 384 numbers can share a collection because the add succeeds. The add succeeds and the search returns nonsense.

**In an interview:** a collection holds vectors of one length from one model. A vector of another length is refused with an error; a vector of the same length from another model is accepted but meaningless next to the others, so switching models means a new collection, re-embedded from the source texts.

**Before you move on**, check that you can:

- build a cosine collection with `embedding_function=None` and add records with ids, vectors, texts and metadata;
- say what `create_collection` and `get_or_create_collection` each do when the name exists;
- rename and delete a collection, and say why the risky calls ran on a scratch copy;
- read the 768-number error and say what `count()` shows after it.

## 3. Upsert

Help-centre articles change: a policy is rewritten, a new article appears. A fresh load usually mixes both kinds of record. **Upsert** is a write that inserts a record when its id is new and replaces the stored record when the id already exists. Its name joins "update" and "insert".

### 3.1 Add an id that exists

`KB_UPDATES` is a batch of 10 invented records with the same keys as `KB`: four changed articles (kb-004, kb-011, kb-019 and kb-027) and six new ones (kb-031 to kb-036). `update_vectors` embeds their texts. The table shows which ids `KB` already holds.

In [ ]:
KB_UPDATES = [                               # invented, like KB: 4 changed articles, then 6 new ones
    {"id": "kb-004", "title": "Deleting your account", "text": "Deleting your account removes your profile, saved addresses and order history after a 30-day grace period. Request it under Settings, Privacy, Delete account; signing in again during the grace period cancels the request.", "category": "account", "product": "web", "updated": 2026},
    {"id": "kb-011", "title": "Updating a payment method", "text": "Add or replace a card or a bank account under Billing, Payment methods. The new method becomes the default for future orders and renewals, and an expired card is removed automatically after its last valid month.", "category": "billing", "product": "web", "updated": 2026},
    {"id": "kb-019", "title": "Returning an item", "text": "Most items can be returned within 60 days of delivery. Start a return under Orders, choose a drop-off point, and show the QR code there; no label or printer is needed.", "category": "shipping", "product": "mobile", "updated": 2026},
    {"id": "kb-027", "title": "Exporting your data", "text": "Request a copy of everything we store about you under Settings, Privacy, Export data, on the website or in the app. We prepare a ZIP file of JSON and CSV files and email a download link within 24 hours.", "category": "technical", "product": "web", "updated": 2026},
    {"id": "kb-031", "title": "Buying a gift card", "text": "Gift cards come in amounts from 10 to 500 dollars and never expire. Buy one under Gift cards, add the recipient's email address, and choose the day the card is delivered.", "category": "billing", "product": "web", "updated": 2026},
    {"id": "kb-032", "title": "Redeeming a gift card", "text": "Enter the 16-character code from a gift card under Wallet, Add gift card in the mobile app. The amount joins your store credit at once and is used before your card at checkout.", "category": "billing", "product": "mobile", "updated": 2026},
    {"id": "kb-033", "title": "Booking a delivery slot", "text": "For large items such as furniture, choose a two-hour delivery window at checkout. We send a text message when the driver is on the way, and you can move the slot once, up to a day before.", "category": "shipping", "product": "web", "updated": 2025},
    {"id": "kb-034", "title": "Using dark mode", "text": "Switch the mobile app to a dark colour scheme under Settings, Display, Theme. Choose Automatic to follow your phone's own setting, so the app turns dark in the evening and light in the morning.", "category": "technical", "product": "mobile", "updated": 2026},
    {"id": "kb-035", "title": "Locked after too many attempts", "text": "After five wrong passwords in a row, sign-in is paused for 15 minutes to protect the account. Wait for the pause to end, or choose Forgot password to set a new one straight away.", "category": "account", "product": "mobile", "updated": 2026},
    {"id": "kb-036", "title": "Reporting a security problem", "text": "If you find a security weakness on the website, write to our security team from the Help menu instead of posting it publicly. We reply within two business days and thank reporters by name if they agree.", "category": "technical", "product": "web", "updated": 2025},
]
update_vectors = embed([a["text"] for a in KB_UPDATES])
known = {a["id"] for a in KB}
print(pd.DataFrame([{"id": a["id"], "title": a["title"], "already in KB": a["id"] in known}
                    for a in KB_UPDATES]))

- The first four rows exist already; the last six are new.
- kb-004's grace period goes from 14 to 30 days, kb-011 adds bank accounts, kb-019's return window goes from 30 to 60 days, kb-027's export now takes 24 hours, and every changed record is marked `updated` 2026.

The next cell sends the whole batch with `add`. `batch` is a dictionary of the four arguments, and `**batch` unpacks it into keyword arguments, so this cell and the next send exactly the same records. `get(ids=["kb-004"])` reads one record back.

**Predict first:** ten records go in with `add`. Six ids are new and four already exist. What will `count()` print, and which grace period will kb-004 show?

In [ ]:
batch = dict(ids=[a["id"] for a in KB_UPDATES], embeddings=update_vectors,
             documents=[a["text"] for a in KB_UPDATES], metadatas=[meta(a) for a in KB_UPDATES])
kb.add(**batch)                               # 6 ids are new, 4 already exist
print("count:", kb.count())
print("kb-004:", kb.get(ids=["kb-004"])["documents"][0])

- `count: 36`, and kb-004 still reads "after a 14-day grace period".
- Chroma's `add` writes the six new ids and ignores the four that exist, without an error and without a warning. Chroma's documentation describes it this way: an add with an existing id is ignored, and a write that replaces must update.
- Nothing told us that four policy changes were lost. The count went up, which looks like success.
- **Common mistake:** reading the count of 36 as proof that the update worked. Read the changed record back: kb-004's text is still the old one.

**In an interview:** in Chroma, `add` with an id that already exists is ignored without an error, so a changed record keeps its old text; only the new ids are written. A load that may meet existing ids uses `upsert`.

### 3.2 Upsert

The same `batch`, sent again with `upsert`.

**Predict first:** what will the count be now, and which grace period will kb-004 show?

In [ ]:
kb.upsert(**batch)                            # the same 10 records
print("count:", kb.count())
print("kb-004:", kb.get(ids=["kb-004"])["documents"][0])

- `count: 36` again, and kb-004 now reads "after a 30-day grace period". The six new ids already existed after 3.1's add, so this upsert replaced all ten records and added none.
- Chroma's `upsert` adds records whose ids are missing and updates those that exist. The three writes side by side:

| Call | The id is new | The id exists |
|---|---|---|
| `add` | inserted | ignored, with no error |
| `update` | an error is logged and nothing changes | replaced |
| `upsert` | inserted | replaced |

- **Common mistake:** using `update` for a regular load. It drops every new article with only a logged error.

Every later section works from one list of the current articles. `KB_ALL` is `KB` with `KB_UPDATES` applied by id: a dictionary keyed by id takes the 30 articles, `by_id.update(...)` replaces the four changed ones and adds the six new ones (the same rule as upsert), and the list comes back in id order. `kb_all_vectors` embeds its 36 texts.

In [ ]:
by_id = {a["id"]: a for a in KB}                     # the 30 articles, keyed by id
by_id.update({a["id"]: a for a in KB_UPDATES})       # the same rule as upsert: 4 replaced, 6 added
KB_ALL = [by_id[i] for i in sorted(by_id)]           # 36 articles in id order
kb_all_vectors = embed([a["text"] for a in KB_ALL])
print(len(KB_ALL), kb_all_vectors.shape)
print(KB_ALL[3]["id"], KB_ALL[3]["updated"], KB_ALL[3]["text"][:60])

- 36 articles and a `(36, 384)` array. The fourth article is kb-004 in its 2026 version.
- Sections 4, 6, 7.3 and 7.4 read only `KB_ALL` and `kb_all_vectors`, so Chroma and Pinecone hold the same vectors and their results can be compared row by row.

**In an interview:** upsert inserts a record whose id is new and replaces one whose id exists, so one call handles a load that mixes new and changed records. `add` ignores existing ids and `update` ignores missing ones.

### 3.3 Stable ids

Upsert decides between insert and replace by the id alone. So the id must name the same article on every load: it comes from the source record (`a["id"]`, such as `"kb-004"`), never from a position in a list. A **stable id** makes a load **idempotent**: running it twice leaves the same records as running it once, so a failed load can simply run again.

The first cell loads the update batch a second time, with the ids from the source.

**Predict first:** the same ten records are upserted again with their own ids. What will the count be?

In [ ]:
ids = [a["id"] for a in KB_UPDATES]     # Try changing it: [f"row-{i}" for i in range(len(KB_UPDATES))]
kb.upsert(ids=ids, embeddings=update_vectors,
          documents=[a["text"] for a in KB_UPDATES], metadatas=[meta(a) for a in KB_UPDATES])
print("count after loading the batch a second time:", kb.count())

- `36`: every id exists, so all ten records were replaced with themselves and nothing was added. Run the cell as often as you like; the count stays 36.

The next cell asks a question the returns article answers, then deletes any record whose id starts `row-`, a tidy-up for the Try changing edit below. Deleting an id the collection does not hold does nothing.

**Predict first:** "how many days do I have to send something back". Which article will come first?

In [ ]:
query_vector = embed(["how many days do I have to send something back"])[0]
print(chroma_hits(kb.query(query_embeddings=[query_vector], n_results=3)))
kb.delete(ids=[f"row-{i}" for i in range(len(KB_UPDATES))])   # removes row ids if the edit made any
print("count after tidying up:", kb.count())

- kb-019 "Returning an item" comes first, at distance 0.464, then kb-015 (a missing order, 0.476) and kb-012 (refund timelines, 0.521). Chroma reports a cosine distance, 1 minus the cosine similarity, so lower is closer.
- The count stays 36 after the tidy-up.
- Try changing it: in the first cell, replace the ids with row numbers, `ids = [f"row-{i}" for i in range(len(KB_UPDATES))]`, and run both cells. The count becomes 46: none of those ids exists, so all ten are inserted beside the 36. The query then returns "Returning an item" twice, kb-019 and row-2, both at 0.464. The tidy-up deletes the row ids and the count returns to 36. Put the source ids back and rerun both cells before section 4.
- **Common mistake:** using a row number or an `enumerate` index as the id. The next load writes the same articles again under ids the collection has never seen, so every article appears twice and searches return duplicates.

**In an interview:** a record's id must come from the source record, so that it names the same item on every load. With stable ids and upsert a load is idempotent; with row-number ids every reload duplicates the data.

**Before you move on**, check that you can:

- predict the count and the text after an `add` of a batch that mixes new and existing ids;
- say what `add`, `update` and `upsert` each do with a new id and with an existing one;
- explain why `KB_ALL` has 36 articles;
- say what row-number ids do to a collection on the second load.

## 4. Partitioning vectors

**Partitioning** splits the stored vectors into separate parts, so that a query searches one part and never sees the others. Chroma partitions with collections. Pinecone partitions with namespaces inside one index (7.3).

### 4.1 A collection per product

The 36 articles go into two collections by product: `kb_web` and `kb_mobile`. Each gets only the rows of `KB_ALL` and `kb_all_vectors` with its product.

**Predict first:** how many articles will each collection hold? (`KB` had 17 web and 13 mobile articles, and the six new ones are three of each.)

In [ ]:
partitions = {}
for product in ["web", "mobile"]:
    rows = [i for i, a in enumerate(KB_ALL) if a["product"] == product]
    part = client.create_collection(
        f"kb_{product}", configuration={"hnsw": {"space": "cosine"}}, embedding_function=None)
    part.add(ids=[KB_ALL[i]["id"] for i in rows], embeddings=kb_all_vectors[rows],
             documents=[KB_ALL[i]["text"] for i in rows], metadatas=[meta(KB_ALL[i]) for i in rows])
    partitions[product] = part
    print(f"kb_{product}: {part.count()} articles")

- `kb_web: 20` and `kb_mobile: 16`, 36 in all.

Now one question for both: "the app keeps closing by itself". The article that answers it, kb-022 "Fixing crashes on phones", is a mobile article.

**Predict first:** which collection finds the crash article, and what will the other one return?

In [ ]:
query_vector = embed(["the app keeps closing by itself"])[0]
for product, part in partitions.items():
    print(f"kb_{product}:")
    print(chroma_hits(part.query(query_embeddings=[query_vector], n_results=3)))

- `kb_mobile` finds kb-022 "Fixing crashes on phones" first, at distance 0.505.
- `kb_web` cannot see it. Its best answer is kb-008 "Ending a plan", a billing article, at 0.729: a cosine similarity of only about 0.27. A partition always returns its nearest records, however far.
- That is the cost of partitioning, shown: a question asked in the wrong partition gets the wrong answers.
- **Common mistake:** reading `kb_web`'s first result as a good match because it came first. Its distance says it is a poor one.

Deleting a partition is one call. The next cell removes the whole mobile partition.

In [ ]:
client.delete_collection("kb_mobile")           # one call removes the whole partition
print([c.name for c in client.list_collections()])

- `kb_mobile` is gone with its 16 records; `kb_articles`, `random_2k` and `kb_web` remain. Removing one customer's data, kept in its own partition, works the same way.

**In an interview:** partitioning keeps groups of vectors apart, so a query in one part never returns records from another, and a whole part is deleted in one call. In Chroma a partition is a collection; in Pinecone it is a namespace.

### 4.2 When to partition

| Reason | Example | What a partition gives |
|---|---|---|
| one customer's data kept apart from another's | a help centre serving fifty client companies | a query in one customer's part never reads another's |
| data that must never mix | articles in English and in French | each question searches its own language only |
| vectors from different models | an old model and its replacement | each part holds one model's vectors; in Pinecone a different size needs a different index, because an index has one dimension |
| deleting a customer | a client company leaves | one call removes all of its records |

One index serving many customers, called **tenants**, is **multitenancy**, the main reason Pinecone's namespaces exist. Pinecone's guide gives each tenant its own namespace for three properties: isolation (a query in one tenant's namespace cannot return another's records), even load (a search's cost follows the size of its namespace, so a large tenant does not slow a small one) and easy removal (deleting a namespace is, in the guide's words, lightweight and almost instant). The free Starter plan allows 100 namespaces per index.

The cost: a question that spans partitions needs one query per partition (Pinecone's `query_namespaces` sends them and merges the results), and every partition is one more name to manage.

**Predict first:** a customer leaves and their data must go. With one partition per customer, how many calls does that take?

<details>
<summary>Check your answer</summary>

One: delete that customer's collection in Chroma, or their namespace in Pinecone. Without partitions you would have to find and delete every one of their records.

</details>

- **Common mistake:** partitioning by something that changes from one question to the next, such as an article's category. A query could then search only one category at a time. That belongs in a metadata filter, section 5.

**In an interview:** I would give each customer their own namespace (or collection): their queries never read another customer's records, a large customer does not slow a small one, and offboarding is deleting one namespace. The price is one query per partition when a question spans them.

**Before you move on**, check that you can:

- split a set of records into one collection per product and count each part;
- say what a query in the wrong partition returns, and how its distance gives that away;
- give two reasons to partition and the cost of partitioning;
- explain multitenancy in two sentences.

## 5. Vector partitioning using metadata

A **metadata filter** is a condition on the fields stored beside each vector, such as category or year, that limits which records a similarity search may return. It carves a part out of one collection at query time instead of storing the parts apart. Both Chroma's `query` and its `get` take one, as `where`.

### 5.1 A where filter

"how do I get my money back", asked of `kb_articles`, first with no filter, then with `where={"category": "billing"}`. A bare value is shorthand: `{"category": "billing"}` means `{"category": {"$eq": "billing"}}`.

**Predict first:** with the billing filter, can a shipping article appear in the top 3?

In [ ]:
query_vector = embed(["how do I get my money back"])[0]
print("no filter:")
print(chroma_hits(kb.query(query_embeddings=[query_vector], n_results=3)))
print('where={"category": "billing"}:')
print(chroma_hits(kb.query(query_embeddings=[query_vector], n_results=3, where={"category": "billing"})))

- With no filter: kb-012 "Refund timelines" (billing) at 0.564, kb-009 "Two entries for one purchase" (billing) at 0.693 and kb-019 "Returning an item" (shipping) at 0.723.
- With the filter, no: every result passes it. kb-012 and kb-009 stay, and the shipping article gives way to kb-014 "Paying in instalments" at 0.740, a weaker match kept because it is a billing article.
- **Common mistake:** reading the filtered results' larger distances as a bug. They are the best matches among the records the filter lets through.

**In an interview:** to restrict a similarity search to one category, add a metadata filter: `where={"category": "billing"}` in Chroma, `filter={"category": {"$eq": "billing"}}` in Pinecone. Every result passes it, even when a closer record does not.

### 5.2 Operators

A filter is a small dictionary. The operators in the cell:

- `$eq` and `$ne` match a field equal, or not equal, to a value.
- `$in` matches one of a list of values of one type.
- `$gte` matches a number at or above the value. `$gt`, `$lt` and `$lte` work the same way, and all four take numbers only, which is why `updated` is stored as an integer year.
- `$and` passes records that pass every filter in its list. `$or` passes records that pass any of them.

Chroma also takes `$nin` (none of a list) and, for metadata that holds a list, `$contains` and `$not_contains`. Each filter object holds one field or one logical operator.

The cell uses `get` with each filter, which returns every record that passes, unranked, and prints the count and the ids.

**Predict first:** how many articles will `{"product": {"$ne": "web"}}` keep? (Section 4 counted 20 web and 16 mobile articles.)

In [ ]:
FILTERS = {
    "$eq":  {"category": {"$eq": "billing"}},
    "$ne":  {"product": {"$ne": "web"}},
    "$in":  {"category": {"$in": ["billing", "shipping"]}},
    "$gte": {"updated": {"$gte": 2026}},
    "$and": {"$and": [{"product": "mobile"}, {"updated": {"$gte": 2025}}]},
}
for name, where in FILTERS.items():
    ids = sorted(kb.get(where=where)["ids"])          # get: every record that passes, unranked
    print(f"{name:5} {len(ids):2} articles: {', '.join(ids)}")

- `$eq` keeps the 9 billing articles, `$ne` the 16 that are not web (all mobile), `$in` the 17 that are billing or shipping, and `$gte` the 15 updated in 2026.
- `$and` keeps the 13 mobile articles updated in 2025 or later.
- **Common mistake:** writing two fields in one filter object, `{"product": "mobile", "updated": {"$gte": 2025}}`. Each object holds one field or one operator; two conditions go in an `$and` list.

The last filter is worth keeping by name. `MOBILE_RECENT` is mobile articles updated in 2025 or later; 7.4 sends it to Pinecone unchanged. Here it ranks the money-back question inside those 13 articles.

In [ ]:
MOBILE_RECENT = {"$and": [{"product": "mobile"}, {"updated": {"$gte": 2025}}]}
print(chroma_hits(kb.query(query_embeddings=[query_vector], n_results=3, where=MOBILE_RECENT)))

- kb-012 "Refund timelines" at 0.564, kb-019 "Returning an item" at 0.723 and kb-014 "Paying in instalments" at 0.740: all three mobile, all updated in 2025 or 2026. kb-009, a web article, is filtered out.

**In an interview:** a filter for mobile articles updated in 2025 or later is `{"$and": [{"product": "mobile"}, {"updated": {"$gte": 2025}}]}`, with the year stored as a number, because the comparison operators take numbers only.

### 5.3 A filter that leaves too few rows

Only a few articles are both billing and mobile. The cell asks for 5 results inside that filter.

**Predict first:** if fewer than 5 records pass the filter, will Chroma raise an error, print a warning, or return fewer?

In [ ]:
BILLING_MOBILE = {"$and": [{"category": "billing"}, {"product": "mobile"}]}
found = kb.query(query_embeddings=[query_vector], n_results=5, where=BILLING_MOBILE)
print(f"asked for 5, got {len(found['ids'][0])}")
print(chroma_hits(found))

- `asked for 5, got 4`: kb-012, kb-014, kb-013 and kb-032, the only billing articles for mobile. Chroma 1.5.9 returns what passes, with no error and no warning.
- Code that reads the fifth result without checking the length fails here with an `IndexError` the first time a filter is tight. Check `len(found["ids"][0])` before reading a position.

The comparison operators take numbers only. The next cell sends the year as text.

In [ ]:
try:
    kb.get(where={"updated": {"$gte": "2025"}})    # the year as text, not as a number
except Exception as err:
    print(f"{type(err).__name__}: {err}")

- `ValueError: Expected operand value to be an int or a float for operator $gte, got 2025 in get.` The text "2025" is a string, so `$gte` refuses it. `2025` without quotes works.
- **Common mistake:** storing a year or a price as text. Every comparison filter on it then raises, and the fix is to store the number.

**In an interview:** a filter that leaves fewer records than `n_results` returns fewer results, silently, so code checks the length of what comes back. Comparison operators such as `$gte` take numbers only.

### 5.4 Partition or filter

| Need | Use | Why |
|---|---|---|
| a hard boundary, such as one customer's data | a namespace or a collection | records outside it are never read, and deleting it removes them all |
| a condition on a field, such as category or year | a metadata filter | one query can combine conditions, and they change from one question to the next |
| both: one customer, recent articles only | a filter inside the customer's namespace | the boundary stays physical; the filter narrows within it |

In Pinecone a filter narrows the candidates while the search still scans the whole namespace, so a scheme that keeps every tenant in one namespace and filters by tenant reads every tenant's data on every search.

**Predict first:** a help centre serves fifty client companies. Which goes in a partition and which in a filter: the company, or the article's category?

<details>
<summary>Check your answer</summary>

The company goes in a partition (a namespace or a collection per company): it is a hard boundary, and a company that leaves is deleted in one call. The category goes in a filter: it changes from one question to the next, and one query can combine it with other conditions.

</details>

- **Common mistake:** putting customers in a filter "because it is one index anyway". Every search then scans every customer's data, and deleting a customer means finding all of their records.

**In an interview:** I use a namespace when the boundary is hard, like one customer's data: the records stay apart and are deleted in one call. I use a metadata filter for conditions that change per question, like category or year, and I can combine the two by filtering inside a namespace.

**Before you move on**, check that you can:

- write a `where` filter with `$eq`, `$in`, `$gte` and `$and`;
- say what comes back when a filter leaves fewer records than `n_results`;
- explain why `{"updated": {"$gte": "2025"}}` raises;
- decide, for a given field, whether it belongs in a partition or a filter.

## 6. Distance metrics

A **distance metric** is the rule that turns two vectors into one number saying how close they are. Every collection and index today was created with one. Three are in common use:

- **Dot product:** multiply matching numbers and add them up. For `[3, 4]` and `[6, 8]`: 3 × 6 + 4 × 8 = 50. Higher is closer, and it grows with the vectors' lengths.
- **Cosine similarity:** the dot product divided by both lengths. A vector's length is the square root of the sum of its squared numbers: for `[3, 4]`, √(3² + 4²) = 5. Cosine runs from -1 to 1, and 1 means the two point exactly the same way, whatever their lengths.
- **Euclidean distance:** the straight-line gap between the two points, week 6's distance: √((3 − 6)² + (4 − 8)²) = √25 = 5. Lower is closer.

### 6.1 Work it by hand first

**Work it by hand first.** Three vectors:

```
a = [3, 4]    b = [6, 8]    c = [4, 3]
```

On paper, for each pair (a, b), (a, c) and (b, c), work out the cosine similarity, the dot product and the Euclidean distance, each rounded to two places. The lengths are 5 for a, 10 for b and 5 for c. Then answer: which pair does each measure call the closest?

<details>
<summary>Check your answers</summary>

| Pair | Cosine | Dot product | Euclidean |
|---|---|---|---|
| a, b | 1.0 | 50 | 5.0 |
| a, c | 0.96 | 24 | about 1.41 |
| b, c | 0.96 | 48 | about 5.39 |

- a, c: the dot product is 3 × 4 + 4 × 3 = 24, the cosine 24 / (5 × 5) = 0.96, and the distance √((3 − 4)² + (4 − 3)²) = √2, about 1.41.
- b, c: the dot product is 6 × 4 + 8 × 3 = 48, the cosine 48 / (10 × 5) = 0.96, and the distance √(2² + 5²) = √29, about 5.39.
- Cosine and dot product call a, b the closest pair. Euclidean calls a, c the closest.
- b is a doubled: it points exactly the same way, which is why the cosine is 1.0, but its tip sits 5 away.

Usual wrong answers:

- A length of 25 for a: the square root was forgotten. The length of a is √25 = 5.
- A cosine above 1, from dividing by only one length. Cosine never exceeds 1.
- "a, b is closest by every measure, since b is just a doubled." Euclidean distance feels the extra length.

</details>

### 6.2 The three in numpy

The same three measures in code: `cosine(u, v)` is `np.dot(u, v) / (np.linalg.norm(u) * np.linalg.norm(v))`, the dot product is `np.dot(u, v)`, and the distance is `np.linalg.norm(u - v)` (`np.linalg.norm` gives a vector's length).

**Predict first:** which pair will Euclidean distance call closest?

In [ ]:
a = np.array([3, 4])
b = np.array([6, 8])                    # Try changing it: b = np.array([30, 40])
c = np.array([4, 3])


def cosine(u, v):
    return np.dot(u, v) / (np.linalg.norm(u) * np.linalg.norm(v))


for name, u, v in [("a,b", a, b), ("a,c", a, c), ("b,c", b, c)]:
    print(f"{name}  cosine {cosine(u, v):.2f}  dot product {np.dot(u, v)}  "
          f"Euclidean {np.linalg.norm(u - v):.2f}")

- `a,b  cosine 1.00  dot product 50  Euclidean 5.00`, `a,c  0.96  24  1.41`, `b,c  0.96  48  5.39`: the hand-worked key.
- Euclidean calls a, c closest, at about 1.41. Cosine and dot product call a, b closest.
- They disagree because of length. Cosine looks only at direction. The dot product and the distance also feel how long the vectors are: b, c share a, c's cosine of 0.96, but b is long, so its dot product is twice as big and its distance much larger.
- Euclidean suits vectors whose length carries meaning, such as counts or measurements.
- Try changing it: `b = np.array([30, 40])`. The cosine of a, b stays 1.00 while the distance grows to 45.00 and the dot product to 250 (3 × 30 + 4 × 40). Put `b = np.array([6, 8])` back and rerun the cell before 6.3.
- **Common mistake:** expecting a longer vector in the same direction to change the cosine. It cannot: cosine ignores length.

**In an interview:** cosine similarity compares direction only; the dot product and Euclidean distance also feel the vectors' lengths. They disagree when lengths differ: `[3, 4]` and `[6, 8]` have a cosine of 1.0 yet sit 5 apart.

### 6.3 Vectors of length 1

A **unit vector** is a vector whose length is 1, so comparing two of them depends only on their direction. `embed` passes `normalize_embeddings=True`, so every vector this week is one. The cell divides a, b and c by their lengths, then prints the three measures again, with the squared Euclidean distance and 2 − 2 × cosine beside it.

For unit vectors two rules hold: the dot product equals the cosine (the lengths it would divide by are 1), and squared Euclidean distance = 2 − 2 × cosine.

**Predict first:** after scaling each vector to length 1, which pair will Euclidean distance call closest now?

In [ ]:
unit = {"a": a / np.linalg.norm(a), "b": b / np.linalg.norm(b), "c": c / np.linalg.norm(c)}
for p, q in [("a", "b"), ("a", "c"), ("b", "c")]:
    u, v = unit[p], unit[q]
    print(f"{p},{q}  cosine {cosine(u, v):.2f}  dot product {np.dot(u, v):.2f}  "
          f"squared Euclidean {np.sum((u - v) ** 2):.2f}  2 - 2 x cosine {2 - 2 * cosine(u, v):.2f}")

- `a,b  1.00  1.00  0.00  0.00`, `a,c  0.96  0.96  0.08  0.08`, `b,c  0.96  0.96  0.08  0.08`. Check one by hand: 2 − 2 × 0.96 = 0.08.
- a and b now coincide: they point the same way, so once both have length 1 they are the same point, 0 apart. Euclidean now agrees with cosine.
- Each measure is a fixed function of the cosine, so the closest pair by one is the closest by all three. For this week's vectors the three metrics rank the articles in the same order.
- **Common mistake:** comparing the plain distance with 2 − 2 × cosine. The rule is for the squared distance, which is what the cell prints.

**In an interview:** for vectors of length 1 the dot product equals the cosine and squared Euclidean distance is 2 − 2 × cosine, so all three rank them in the same order. That is why normalised embeddings make the choice of metric change the numbers but not the ranking.

### 6.4 Three collections, one set of vectors

Chroma's three spaces are all distances, so lower is closer:

- `cosine`: 1 − cosine similarity.
- `ip` (inner product, another name for the dot product): 1 − the dot product.
- `l2`: the squared Euclidean distance, Σ(Aᵢ − Bᵢ)², not the plain one.

The cell stores the 36 articles three times, in `kb_cosine`, `kb_ip` and `kb_l2`, and asks each the money-back question.

**Predict first:** the same unit vectors go into all three. Which two collections will print the same distances, and what will the third print?

In [ ]:
query_vector = embed(["how do I get my money back"])[0]
table = {}
for space in ["cosine", "ip", "l2"]:
    col = client.create_collection(
        f"kb_{space}", configuration={"hnsw": {"space": space}}, embedding_function=None)
    col.add(ids=[a["id"] for a in KB_ALL], embeddings=kb_all_vectors,
            documents=[a["text"] for a in KB_ALL], metadatas=[meta(a) for a in KB_ALL])
    found = col.query(query_embeddings=[query_vector], n_results=3)
    table[f"kb_{space}"] = [f"{i}  {d:.3f}" for i, d in zip(found["ids"][0], found["distances"][0])]
print(pd.DataFrame(table, index=["1st", "2nd", "3rd"]))

- The same top 3 in all three: kb-012, kb-009 and kb-019.
- `kb_cosine` and `kb_ip` print the same distances, 0.564, 0.693 and 0.723: for unit vectors 1 − dot equals 1 − cosine.
- `kb_l2` prints exactly double, 1.127, 1.386 and 1.446: 2 − 2 × cosine is twice 1 − cosine.
- **Common mistake:** reading `kb_l2`'s 1.127 as a worse match than `kb_cosine`'s 0.564. They are two scales for the same closeness, and the order is the same.

**In an interview:** Chroma's `cosine`, `ip` and `l2` spaces are all distances, lower closer. On unit vectors `cosine` and `ip` give identical numbers and `l2` (squared) gives exactly double, with the same ranking.

### 6.5 Choosing a metric

- Use the metric the embedding model was trained with. all-MiniLM-L6-v2 was fine-tuned on the cosine similarity of sentence pairs, so cosine is its metric. Cosine on raw vectors and dot product on normalised vectors are the same calculation, so a dot-product index also suits it when the vectors have length 1.
- The model card says which; Pinecone's index guide says the index's dimension and metric should match the model's. Most text models name cosine.
- Euclidean suits vectors that encode counts or measurements, where length carries meaning.
- The metric is set once, at creation: `configuration={"hnsw": {"space": "ip"}}` in Chroma, `metric="euclidean"` in Pinecone's `create_index`. Neither database can change it later, so a wrong choice means a new collection or index, filled again.

**Predict first:** a new text model's card says it was trained with cosine similarity. Which metric does its index get, and when is it set?

<details>
<summary>Check your answer</summary>

Cosine (or dot product, if every vector is normalised to length 1), set when the index or collection is created. It cannot be changed afterwards.

</details>

- **Common mistake:** picking Euclidean because it is the familiar distance. For a text model the model card names the metric.

**In an interview:** I pick the metric the embedding model was trained with, from its model card; for most text models that is cosine, or dot product on normalised vectors. I set it when I create the index, because neither Chroma nor Pinecone can change it later.

**Before you move on**, check that you can:

- compute cosine, dot product and Euclidean distance for two short vectors by hand;
- say which pair of a, b and c each measure calls closest, and why they disagree;
- show from 2 − 2 × cosine why the three agree on unit vectors;
- predict the distances a `cosine`, an `ip` and an `l2` collection print for the same unit vectors.

## 7. The same in Pinecone

Everything so far ran in Chroma, inside the notebook. Pinecone is hosted: the index lives on Pinecone's servers, in your own free project. This section repeats the day's ideas there. Every cell uses the pinned client 9.1.0. Pinecone's current documentation leads with a newer style of calls (a Documents API, `pc.indexes.create(schema=..., deployment=...)`) that this client does not use; copy calls from this notebook, not from those pages.

### 7.1 Index management

The key is read from Colab's Secrets panel straight into the client, and no cell prints it, stores it in a variable or writes it anywhere. Two Colab errors can stop this cell:

- `SecretNotFoundError`: no secret by that name is visible. Open the Secrets panel (the key icon in the left bar) and add `PINECONE_API_KEY`, spelled exactly.
- `NotebookAccessError`: the secret exists but its Notebook access switch is off. Switch it on.

The first time a notebook reads the secret, Colab may instead show a dialog asking you to grant it access. Unanswered for about a minute, the cell fails with `TimeoutException` and Run all skips the rest: click **Grant access** and run the cell again, or switch on Notebook access before you run section 7.

The cell prints how many indexes the project holds, not their names, so an index you made for something else never appears on screen.

**Predict first:** your project held no index before today. What will the count be?

In [ ]:
if IN_COLAB:
    from google.colab import userdata
    pc = Pinecone(api_key=userdata.get("PINECONE_API_KEY"))   # from Secrets straight into the client
else:
    pc = Pinecone()   # outside Colab only: the client reads the PINECONE_API_KEY environment variable
names = pc.list_indexes().names()                           # the first call that reaches Pinecone
print(f"connected: {len(names)} index(es) in this project; kb-wednesday among them: {'kb-wednesday' in names}")

- On a project used only for this course the count is 0, and `kb-wednesday` is not there yet. Tuesday's notebook deleted `kb-tuesday` at its end.

An index's lifecycle in the SDK is create, describe, list, stats, configure and delete. The create cell checks `pc.has_index` first, so a rerun skips it. `dimension=384` and `metric="cosine"` match the embedding model. `ServerlessSpec(cloud="aws", region="us-east-1")` asks for a serverless index in the one region the free Starter plan offers. `create_index` waits until the index is ready.

**Predict first:** which metric and dimension will `describe_index` print?

In [ ]:
if not pc.has_index("kb-wednesday"):                   # a rerun skips the create
    pc.create_index(name="kb-wednesday", dimension=384, metric="cosine",
                    spec=ServerlessSpec(cloud="aws", region="us-east-1"))
info = pc.describe_index("kb-wednesday")
print("dimension:", info.dimension)
print("metric:", info.metric)
print("host:", info.host)
print("status:", info.status)
print("deletion protection:", info.deletion_protection)
print("index count:", len(pc.list_indexes().names()))

- `dimension: 384` and `metric: cosine`, as created. `host` is the address every data call for this index goes to. `status` reads `ready=True, state='Ready'` once the index takes requests, and `deletion protection` is `disabled`.
- The index count went up by one.
- These are the 9.1.0 SDK's fields. Pinecone's newest REST API names its raw response differently; the SDK keeps these.
- The Pinecone console at app.pinecone.io runs the same lifecycle by hand: a create-index form, a delete button, and an index's backups under the Backups tab or the Indexes tab. The SDK is what a program uses.

`pc.Index("kb-wednesday")` returns the handle every data call goes through. `describe_index_stats()` reports the vectors it holds.

In [ ]:
index = pc.Index("kb-wednesday")                        # the handle every data call goes through
print(index.describe_index_stats())

- `dimension=384, total_vector_count=0, metric='cosine', namespaces=0`: a new index holds nothing yet.

**Deletion protection** is an index setting that refuses every delete request until it is switched off again. `configure_index` switches it; it also changes tags, replicas, the pod type and read capacity, and it has no field for the metric or the dimension. The cell switches protection on, tries a delete inside `try`/`except`, then switches it off.

**Predict first:** deletion protection is on, and the cell calls `delete_index`. What happens?

In [ ]:
pc.configure_index("kb-wednesday", deletion_protection="enabled")
print("deletion protection:", pc.describe_index("kb-wednesday").deletion_protection)
try:
    pc.delete_index("kb-wednesday")
except Exception as err:
    print(f"refused: {type(err).__name__}: {err}")
pc.configure_index("kb-wednesday", deletion_protection="disabled")
print("deletion protection:", pc.describe_index("kb-wednesday").deletion_protection,
      "| index still there:", pc.has_index("kb-wednesday"))

- `refused: ForbiddenError: [403 FORBIDDEN] Deletion protection is enabled for this index. Disable deletion protection before retrying.` 403 is the HTTP status a server sends when it understands a request and refuses it.
- After `"disabled"` the index is still there, and 7.6 can delete it.
- The free Starter plan (Pinecone's pricing page, fetched 2026-11-24): up to 5 indexes, 2 GB of storage, 1M read units and 2M write units a month, 100 namespaces per index, AWS us-east-1 only, no backups. Usage past an allowance is blocked, not billed. Today keeps far inside it: at most two indexes at once, 72 vectors at most in one.
- **Common mistake:** stopping between the two `configure_index` lines. Protection stays on and 7.6's delete is refused with a 403. Rerun this cell, then 7.6.

**In an interview:** with the SDK I check `has_index`, then `create_index` with the model's dimension and metric and a serverless spec; `describe_index` shows the dimension, metric, host, status and protection, and `list_indexes` the names. With deletion protection on, `delete_index` fails with a 403 until `configure_index` switches it off.

### 7.2 Upsert

Pinecone has no separate add: every write of a record is an upsert. A new id is inserted, and an existing id has its entire record overwritten, vector and metadata.

- A record is `{"id": ..., "values": ..., "metadata": ...}`. `to_records` builds them: `values` is the vector as a plain list (`.tolist()`; Pinecone's client cannot send numpy numbers), and the metadata is `meta(a)` plus the text. `{**meta(a), "text": ...}` copies one dictionary's keys into a new one and adds a key.
- The 30 `KB` articles go in first, then the 10 records of `KB_UPDATES`.
- Pinecone's counts lag a write by a few seconds, so `wait_for_count` polls the stats every 2 seconds for up to 60 seconds until they show the expected number.

**Predict first:** Chroma's `add` in 3.1 skipped the four existing ids. What will Pinecone's upsert of the same ten records do, and what will the count reach?

In [ ]:
def to_records(articles, vectors):
    """Pinecone records: the id, the vector as a plain list, and the metadata with the text."""
    return [{"id": a["id"], "values": v.tolist(), "metadata": {**meta(a), "text": a["text"]}}
            for a, v in zip(articles, vectors)]


index.upsert(vectors=to_records(KB, kb_vectors))                  # 30 new ids
wait_for_count(index, 30)
index.upsert(vectors=to_records(KB_UPDATES, update_vectors))      # 6 new ids, 4 that exist
wait_for_count(index, 36)

- 30, then 36: the six new ids inserted and the four existing ones overwritten. In Chroma the count also reached 36 after the add, but there the four changes were lost. The fetch below shows that here they were not.

`index.fetch(ids=["kb-004"])` returns `vectors`, a map from each id to its stored record. A fetch, like a count, can lag a write by a moment, so the cell fetches again until it sees the 2026 version.

In [ ]:
for attempt in range(10):                       # a fetch can lag the upsert by a moment
    record = index.fetch(ids=["kb-004"]).vectors["kb-004"]
    if record.metadata["updated"] == 2026:
        break
    time.sleep(1)
print("kb-004:", record.metadata["text"])
print("updated:", record.metadata["updated"], "| vector length:", len(record.values))
stats = index.describe_index_stats()
print("namespaces:", {name: s.vector_count for name, s in stats.namespaces.items()})

- kb-004 reads "after a 30-day grace period", `updated: 2026`, with its vector of 384 numbers: the upsert replaced the whole record.
- No namespace was named, so the records went to the **default namespace**, which the stats list as `'__default__'`.
- One upsert request should hold up to 1,000 records within 2 MB; 9.1.0's `upsert` sends large batches four at a time by default. Batched loads are Thursday's.
- **Common mistake:** reading the count right after an upsert, seeing the old number and upserting again. The count lags by a few seconds; `wait_for_count` waits for it.
- **Common mistake:** upserting a record with only new metadata to change one field. Upsert overwrites the entire record; a change to part of one belongs in `update` (Tuesday's 9.3).

**In an interview:** in Pinecone every write is an upsert: a new id is inserted and an existing id has its whole record overwritten. Counts and fetches catch up a few seconds after a write, so a load waits for the count before reading back.

### 7.3 Namespaces

A **Pinecone namespace** is a named partition inside an index. Every upsert, query and other read or write targets exactly one namespace, so records in other namespaces are never seen. A namespace is created by the first upsert into it.

The cell upserts the 36 articles of `KB_ALL` again, by product, into the namespaces `web` and `mobile`, and waits for each one's count. The default namespace keeps its 36.

**Predict first:** after the upserts, how many namespaces will the stats list, and how many vectors in all?

In [ ]:
for product in ["web", "mobile"]:
    rows = [i for i, a in enumerate(KB_ALL) if a["product"] == product]
    index.upsert(vectors=to_records([KB_ALL[i] for i in rows], kb_all_vectors[rows]), namespace=product)
    wait_for_count(index, len(rows), namespace=product)
stats = index.describe_index_stats()
for name, summary in stats.namespaces.items():
    print(f"namespace {name!r}: {summary.vector_count} vectors")
print("total:", stats.total_vector_count)

- Three namespaces: `'__default__'` with 36, `'mobile'` with 16 and `'web'` with 20, and 72 vectors in all. The same article now sits in two namespaces; they never meet in a query.
- Other namespace calls: `index.list_namespaces()` and `index.describe_namespace(...)` list and describe them, and `index.delete_namespace(namespace="mobile")` deletes every record in one namespace, permanently: offboarding a tenant in one call.

The query names its namespace. Each match carries an `id`, a `score` and the metadata (with `include_metadata=True`).

**Predict first:** the crash question is asked in the `mobile` namespace. Which article comes first, and roughly what score will it get, given Chroma's distance of 0.505 for it in 4.1?

In [ ]:
query_vector = embed(["the app keeps closing by itself"])[0]
found = index.query(vector=query_vector.tolist(), top_k=3, namespace="mobile",   # Try changing it: "web"
                    include_metadata=True)
print(pinecone_hits(found))

- kb-022 "Fixing crashes on phones" first, score 0.495, then kb-005 (0.253) and kb-034 (0.238): the same three as Chroma's `kb_mobile` in 4.1.
- Pinecone reports a **score**: on a cosine index it is the cosine similarity, so higher is closer. Chroma reported a distance, 1 minus the similarity, lower closer. 1 − 0.505 = 0.495.
- Try changing it: `namespace="web"` in the query, keeping the crash question. Only web articles come back, with low scores: kb-008 "Ending a plan" at about 0.27, then kb-036 and kb-027 at about 0.23 and 0.22, the same three as Chroma's `kb_web`. Put `namespace="mobile"` back and rerun the cell before 7.4.
- **Common mistake:** reading a low score in the wrong namespace as an error. The namespace did its job; the question was asked in the wrong partition.

**In an interview:** a Pinecone namespace is a named partition inside an index; every write and query targets one, so one customer's query never reads another's records, and `delete_namespace` removes a customer. A question across namespaces needs one query each, or `query_namespaces`.

### 7.4 Metadata filters

Pinecone's `filter` takes the same dictionary syntax as Chroma's `where`: `$eq`, `$ne`, `$gt`, `$gte`, `$lt`, `$lte` (numbers), `$in` and `$nin`, `$and` and `$or`, plus `$exists` and `$not`. A metadata field can also hold a list of strings, and a filter on it matches when any element matches (Thursday uses that).

The cell asks the money-back question in the default namespace twice: with `{"category": {"$eq": "billing"}}`, and with `MOBILE_RECENT` from 5.2, unchanged.

**Predict first:** will 5.2's `MOBILE_RECENT` need any change to run in Pinecone, and which three ids will it return?

In [ ]:
query_vector = embed(["how do I get my money back"])[0]
print('filter={"category": {"$eq": "billing"}}:')
print(pinecone_hits(index.query(vector=query_vector.tolist(), top_k=3, include_metadata=True,
                                filter={"category": {"$eq": "billing"}})))
print("filter=MOBILE_RECENT, unchanged from 5.2:")
print(pinecone_hits(index.query(vector=query_vector.tolist(), top_k=3, include_metadata=True,
                                filter=MOBILE_RECENT)))

- The billing filter returns kb-012 (0.436), kb-009 (0.307) and kb-014 (0.260): the ids of 5.1's billing query, and each score is 1 minus Chroma's distance (1 − 0.564 = 0.436).
- `MOBILE_RECENT` runs unchanged and returns kb-012, kb-019 and kb-014, as in 5.2.
- The same vectors and the same metadata give the same answers in both databases.
- A filter narrows the candidates, but the search still scans the whole namespace, so a hard boundary belongs in a namespace (5.4).
- **Common mistake:** querying a namespace that holds other records and expecting 5.2's ids. A filter narrows only within the namespace queried.

**In an interview:** a Pinecone metadata filter is `index.query(vector=..., top_k=3, filter={"category": {"$eq": "billing"}}, include_metadata=True)`; the operators and the dictionary shape are the same as Chroma's `where`.

### 7.5 The metric is fixed at creation

*Going deeper. The one idea to keep: an index's metric is set when it is created, so a different metric means a new index.*

`configure_index` has no metric field. To try Euclidean, we create a second index, `kb-wednesday-l2`, with `metric="euclidean"`, fill it with the same 36 vectors and ask the same money-back question. On a Euclidean index Pinecone's score is the squared Euclidean distance, and the best match has the lowest score. Two indexes at once is today's limit; the Starter plan allows five.

In [ ]:
if not pc.has_index("kb-wednesday-l2"):
    pc.create_index(name="kb-wednesday-l2", dimension=384, metric="euclidean",
                    spec=ServerlessSpec(cloud="aws", region="us-east-1"))
index_l2 = pc.Index("kb-wednesday-l2")
index_l2.upsert(vectors=to_records(KB_ALL, kb_all_vectors))
wait_for_count(index_l2, 36)

In [ ]:
cos = pinecone_hits(index.query(vector=query_vector.tolist(), top_k=3, include_metadata=True))
l2 = pinecone_hits(index_l2.query(vector=query_vector.tolist(), top_k=3, include_metadata=True))
print(pd.DataFrame({"cosine id": cos["id"], "cosine score": cos["score"],
                    "2 - 2 x cosine": (2 - 2 * cos["score"]).round(3),
                    "euclidean id": l2["id"], "euclidean score": l2["score"]}))

- The same three ids in the same order, kb-012, kb-009 and kb-019. The Euclidean scores are 2 − 2 × the cosine scores: kb-012's cosine score of about 0.436 becomes about 1.13, and the smallest number comes first.
- The scores match Chroma's `l2` distances in 6.4 for the same reason: both are squared distances on unit vectors.
- A `dotproduct` index would score the dot product, the more positive the closer; on unit vectors that equals the cosine score.

### 7.6 Clean up

Each of today's indexes is deleted if it exists, with the same `has_index` guard the create cells use, so the cell raises no error when 7.5 was skipped and can safely run twice. Each project is then empty between days, a rerun starts clean, and the week stays far inside the free plan.

**Predict first:** 7.5 was skipped. Will deleting `kb-wednesday-l2` raise an error?

In [ ]:
for name in ["kb-wednesday", "kb-wednesday-l2"]:
    if pc.has_index(name):                    # the same guard the create cells use
        pc.delete_index(name)
        print("deleted", name)
names = pc.list_indexes().names()
print("kb-wednesday left:", "kb-wednesday" in names, "| kb-wednesday-l2 left:", "kb-wednesday-l2" in names)

- `deleted kb-wednesday`, `deleted kb-wednesday-l2` (only the first when 7.5 was skipped: the guard skips a name that does not exist, with no error), then both reported as not left.
- A delete only works with deletion protection off, which 7.1 switched back to `"disabled"`.
- Backups, which would let you restore a deleted index, are not on the free Starter plan; the notes page covers backups and a rollback plan.

**In an interview:** at the end of a job I delete the indexes I created, each behind `has_index`, so the clean-up never fails on a missing index and can run twice. On a paid plan I would take a backup first.

**Before you move on**, check that you can:

- connect with the key from Secrets, and say what `SecretNotFoundError` and `NotebookAccessError` each mean;
- create, describe, protect and delete a Pinecone index, and read the 403 refusal;
- say what Pinecone's upsert does with an existing id, and why the cells wait for the count;
- upsert into two namespaces, query one, and turn a Chroma distance into the matching Pinecone score.

## Recap

| Idea | What it is for | Watch out for |
|---|---|---|
| Vector index and HNSW | finding near vectors without comparing the query with every one | approximate: a few true neighbours can be missed; `space`, `ef_construction` and `max_neighbors` are fixed at creation |
| Collection | one model's vectors of one size, with ids, texts and metadata | `create_collection` raises on an existing name; `delete_collection` cannot be undone; a 768-number vector is refused by a 384-number collection |
| Upsert | one write for a load that mixes new and changed records | Chroma's `add` silently skips existing ids; ids come from the source, never a row number |
| Partition (collection or namespace) | keeping data apart, such as one customer's, and deleting it in one call | a query reads one partition only; a question across partitions needs a call each |
| Metadata filter | narrowing a search by a field that changes per question | comparison operators take numbers only; a tight filter returns fewer results with no warning |
| Distance metric | ranking vectors; set from the model card | Chroma reports distances (lower closer), Pinecone's cosine a score (higher closer) and its euclidean a squared distance (lower closer); on unit vectors all three rank alike |
| Pinecone index management | the same lifecycle on a hosted index | the metric is fixed at creation; deletion protection refuses deletes; delete every index at the end |

**Quick check**
1. What is a vector index for, and why is an HNSW index approximate?
2. What is the difference between `create_collection` and `get_or_create_collection`, and what happens when you add a 768-number vector to a 384-number collection?
3. A nightly job writes the help-centre articles to Chroma with `add`, using each row number as its id. What goes wrong, and what is the fix?
4. Walk me through a Pinecone index's lifecycle with the SDK. What happens if someone deletes it while deletion protection is on?
5. How would you keep two customers' data apart in one Pinecone index, and when would you use a metadata filter instead?
6. Why do cosine, dot product and Euclidean distance rank this week's vectors in the same order, and when would they disagree?

<details>
<summary>Answers (say yours aloud first)</summary>

1. Without an index a search compares the query with every stored vector, so its time grows roughly in step with the rows, as 1.1 showed. HNSW links each vector to a few near neighbours in layers and walks from the top layer down, so the cost grows about logarithmically. The walk is greedy and can stop at a local minimum, so it sometimes misses a true neighbour; 1.4 measured a recall of about 0.9 on random vectors.
2. `create_collection` raises if the name exists; `get_or_create_collection` returns the existing collection and ignores its other arguments. The 768-number add raises an error naming both sizes ("expecting embedding with dimension of 384, got 768"), and the count does not change.
3. `add` ignores ids that already exist, so changed articles silently keep their old text. Row-number ids write the same article under a new id whenever the list changes, so searches return duplicates. Use each article's own id and `upsert`, which makes the load idempotent.
4. Check `has_index`, then `create_index` with the model's dimension and metric and a serverless spec. `describe_index` shows the dimension, metric, host, status and deletion protection; `list_indexes` the names; `describe_index_stats` the vector counts per namespace. With protection switched on by `configure_index`, `delete_index` fails with a 403 until protection is disabled; then delete it behind the same `has_index` guard.
5. A namespace per customer: every upsert and query names it, one customer's query never reads the other's records, and `delete_namespace` removes a customer in one call. A filter is for conditions that change per question, such as category or year, because it still scans the whole namespace; the two combine as a filter inside the customer's namespace.
6. `embed` gives every vector length 1. Then the dot product equals the cosine and the squared distance is 2 − 2 × cosine, so each is a fixed function of the cosine and the order is the same. They disagree when lengths differ: `[3, 4]` and `[6, 8]` have a cosine of 1.0 but sit 5 apart. Pick the metric the model was trained with, at creation.

</details>